# quant-lens: Gemma 2 2B

The same experiment on Gemma 2 2B with Gemma Scope residual SAEs
(`gemma-scope-2b-pt-res-canonical`, `layer_{layer}/width_16k/canonical`, hook
`blocks.{layer}.hook_resid_post`, layers 5, 12 and 19 by default from the registry).

Runtime > Change runtime type > **A100 GPU**. The float32 reference model plus one quantized copy
take roughly 21 GB, which is why this notebook uses fewer sequences and a smaller batch than the
GPT-2 one.

## Hugging Face login

Gemma 2 is gated. First accept the license on
[huggingface.co/google/gemma-2-2b](https://huggingface.co/google/gemma-2-2b) with your account,
then log in here with a token that has read access.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
!git clone https://github.com/iamartyaa/quant-lens.git
%cd quant-lens
!pip install -q -e .

## RTN, with the extra analyses

Ablation, fragility and steering run at `--analysis-bits` (default: the lowest of `--bits`).
Ablation quantizes one block at a time, so it adds one pass per transformer block.

In [ ]:
!python -m quant_lens.experiment --model gemma-2-2b --scheme rtn \
    --bits 8 6 4 3 --n-seqs 64 --batch-size 2 \
    --save-per-feature --ablate-layers --fragility --steering \
    --out results/gemma_rtn

## AWQ-lite

In [ ]:
!python -m quant_lens.experiment --model gemma-2-2b --scheme awq_lite \
    --bits 8 6 4 3 --n-seqs 64 --batch-size 2 --calib-seqs 32 \
    --save-per-feature --out results/gemma_awq_lite

## Reports

In [ ]:
RUN_DIRS = ['results/gemma_rtn', 'results/gemma_awq_lite']
for d in RUN_DIRS:
    !python -m quant_lens.report {d}

In [ ]:
import os
import pandas as pd
from IPython.display import Image, Markdown, display

COLS = ['hook', 'bits', 'scheme', 'group_size', 'median_feature_corr', 'jaccard', 'topk_overlap',
        'lost_frac', 'new_frac', 'fve_ref', 'fve_q', 'weight_rel_err', 'logit_kl', 'top1_agree']
IMAGES = ['summary.png', 'ablation_heatmap.png', 'fragility_scatter.png', 'steering.png']

for d in RUN_DIRS:
    if not os.path.exists(f'{d}/summary.csv'):
        print(f'{d}: no summary.csv (run skipped or failed)')
        continue
    display(Markdown(f'## {d}'))
    df = pd.read_csv(f'{d}/summary.csv')
    display(df[[c for c in COLS if c in df.columns]].round(4))
    for name in ['ablation_table.csv', 'fragility_summary.csv', 'steering.csv']:
        if os.path.exists(f'{d}/{name}'):
            display(Markdown(f'**{name}**'))
            display(pd.read_csv(f'{d}/{name}').round(4))
    for img in IMAGES:
        if os.path.exists(f'{d}/{img}'):
            display(Image(f'{d}/{img}'))